# SOMP Attack Walkthrough

This notebook dissects the SOMP pipeline. Unlike DAGER, SOMP does not only keep per-position token candidates. It builds a broader global token pool, decodes sequence candidates, clusters them, and then uses matching pursuit to explain the mixed gradient.

We will inspect:

1. global candidate-pool size
2. per-length candidate construction
3. beam-decoded candidates
4. clustered representatives
5. final predictions

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
from notebooks.attack_walkthrough_helpers import (
    SMALL_SENTENCES,
    LONG_SENTENCES,
    build_attack_args,
    make_labels,
    trace_somp_attack,
)

pd.set_option('display.max_colwidth', 200)

## Small Sentences

In [ ]:
small_sentences = SMALL_SENTENCES
small_args = build_attack_args(
    dataset='sst2',
    batch_size=len(small_sentences),
    model_path='gpt2',
    extra_args=['--rank_tol', '1e-8', '--max_ids', '64'],
)
small_labels = make_labels(len(small_sentences), small_args.device, label=0)
small_trace = trace_somp_attack(small_args, small_sentences, labels=small_labels)

print('Rank:', small_trace['rank'])
print('Global candidate pool size:', small_trace['candidate_pool_size'])
small_trace['length_table']

In [ ]:
small_trace['candidate_pool_preview']

In [ ]:
small_trace['clustered_preview']

In [ ]:
small_trace['predictions']

## Long Sentences

In [ ]:
long_sentences = LONG_SENTENCES
long_args = build_attack_args(
    dataset='sst2',
    batch_size=len(long_sentences),
    model_path='gpt2',
    extra_args=['--rank_tol', '1e-8', '--max_ids', '64'],
)
long_labels = make_labels(len(long_sentences), long_args.device, label=0)
long_trace = trace_somp_attack(long_args, long_sentences, labels=long_labels)

print('Rank:', long_trace['rank'])
print('Global candidate pool size:', long_trace['candidate_pool_size'])
long_trace['length_table']

In [ ]:
long_trace['candidate_pool_preview']

In [ ]:
long_trace['clustered_preview']

In [ ]:
long_trace['predictions']

## What To Look For

- `candidate_pool_size` tells you how broad the initial search is.
- `length_table` shows how many positions stayed alive for each target length.
- `candidate_pool_preview` is the raw decoded sequence pool before clustering.
- `clustered_preview` is what survives after near-duplicate pruning.
- The final `predictions` come after the OMP-style residual matching stage.